In [1]:
%load_ext autoreload
%autoreload 2
%env ANYWIDGET_HMR=1

env: ANYWIDGET_HMR=1


In [2]:
import asyncio
import pathlib
import time

import ipywidgets as widgets
import numpy as np

from maia_waterfall_widget import Waterfall, WaterfallShape

In [3]:
w_shape = WaterfallShape(time=300, freq=400)
w = Waterfall(w_shape, layout=widgets.Layout(height="37vh"))
w.mqtt_url = "ws://mahali-sdr-rx-009:9001"
w.mqtt_topic = "dt/vsword/spectrogram-survey/+/data"
w

In [4]:
kwargs = {
    "style": {"description_width": "initial"},
}
spectrum_visible = widgets.Checkbox(
    value=False,
    description="Show spectrum",
    **kwargs,
)
widgets.jslink((spectrum_visible, 'value'), (w, 'spectrum_visible'))
waterfall_visible = widgets.Checkbox(
    value=True,
    description="Show waterfall",
    **kwargs,
)
widgets.jslink((waterfall_visible, 'value'), (w, 'waterfall_visible'))
cmap_select = widgets.Dropdown(
    options=['viridis', 'turbo', 'inferno'],
    value='turbo',
    description="Colormap:",
    **kwargs,
)
widgets.link((cmap_select, 'value'), (w, 'colormap'))
min_max_db = widgets.IntRangeSlider(
    value=[-75, -5],
    min=-120,
    max=10,
    step=1,
    continuous_update=False,
    description="Scale (dB):",
    layout=widgets.Layout(flex='4 1 30%'),
    **kwargs,
)
widgets.link((min_max_db, 'value'), (w, 'waterfall_min_db'), transform=[
    lambda src: src[0],
    lambda tgt: (tgt, min_max_db.value[1]),
])
widgets.link((min_max_db, 'value'), (w, 'waterfall_max_db'), transform=[
    lambda src: src[1],
    lambda tgt: (min_max_db.value[0], tgt),
])
subchannel_idx = widgets.IntText(
    value=0,
    description="Subchannel index:",
    **kwargs,
)
widgets.jslink((subchannel_idx, 'value'), (w, 'subchannel_idx'))
show_toggles = widgets.VBox(
    [waterfall_visible, spectrum_visible],
    layout=widgets.Layout(flex='1 1 10%'),
)
entries = widgets.VBox(
    [cmap_select, subchannel_idx],
    layout=widgets.Layout(flex='2 1 20%'),
)
widgets.HBox([show_toggles, entries, min_max_db], layout=widgets.Layout(width="100%"))